# Interaction SimPy <> Gym

Consider the following simple manufacturing scenario:

A machine processes jobs but can fail as it degrades over time. When the machine is new the failure is minor, when it is already worn, the failure is major. When it fails, a decision must be made: Either perform a quick, cheap repair (that restores the state to worn) or a thorough, expensive overhaul (that restores the state to new). The different actions take different times to complete and have different costs. 

The MDP is defined as follows:
- There are three states that indicate the degradation of the machine: *New*, *Worn* and *Broken*
- There are two actions: 
    - *Repair*, that restores the machine to *Worn* or 
    - *Overhaul*, that restores to machine to *New*
- We don't know transition probabilities or reward functions (from the agent's POV)

We will consider minimizing the total operational cost (downtime + repair costs) over time as the goal.


## The simpy environment
First, let's recreate the environment in simpy, using a simple event-based model.
We will sample the time to failure for the two states new and worn from an exponential distribution (this is the probability distribution for time to failure in a Poisson process).

In [59]:
import simpy
import random

class MachineEnv:
    def __init__(self, env: simpy.Environment):
        self.env = env
        self.broken = self.env.event()
        self.repaired = self.env.event()

        self.env.process(self.machine_process())
        self.env.process(self.decision_process())

    # --- 1. The Machine Process ---
    def machine_process(self):
        """Simulates the machine's lifecycle of operating and breaking."""
        machine_state = 'New'
        params = {'New': 100, 'Worn': 40} # Time to failure for each state

        while True:
            # Operate until the next failure
            time_to_failure = random.expovariate(1.0 / params[machine_state])
            print(f"Time {self.env.now:.2f}: Machine is '{machine_state}'. Operating for {time_to_failure:.2f} hours.")
            yield self.env.timeout(time_to_failure)
            
            # --- Failure Occurs ---
            failure_type = 'Minor_Failure' if machine_state == 'New' else 'Major_Failure'
            print(f"Time {self.env.now:.2f}: --- Machine FAILED ({failure_type}). Signaling for a decision. ---")
            self.broken.succeed(value=failure_type) # Signal that the machine is broken
            self.broken = self.env.event() # Reset the event for the next failure

            # Wait for the decision process to send back a repair action
            chosen_action = yield self.repaired
            print(f"Time {self.env.now:.2f}: {chosen_action} completed.")
            
            # Update state based on the repair action
            machine_state = 'Worn' if chosen_action == 'Quick_Fix' else 'New'

    # --- 2. The Decision Process (Our temporary "Agent") ---
    def decision_process(self):
        """Waits for the machine to break, then makes a maintenance decision."""
        params = {'Quick_Fix': 10, 'Overhaul': 25} # Repair durations

        while True:
            # Wait for the machine to signal that it's broken
            yield self.broken
            
            # --- Make a Decision ---
            # For now, a simple rule: always overhaul
            decision = 'Overhaul' 
            print(f"Time {self.env.now:.2f}: Decision process sees failure. Choosing '{decision}'. Will take {params[decision]} hours to repair.")
            
            # Simulate the time it takes to perform the repair
            yield self.env.timeout(params[decision])
            
            # Signal to the machine what repair was done so it can continue
            self.repaired.succeed(value=decision)
            self.repaired = self.env.event() # reset repair event for the next decision


# Run the simulation for a fixed duration
print("--- Starting Simulation ---")
env = simpy.Environment()
machine = MachineEnv(env)
env.run(until=500)
print("--- Simulation Finished ---")

--- Starting Simulation ---
Time 0.00: Machine is 'New'. Operating for 99.77 hours.
Time 99.77: --- Machine FAILED (Minor_Failure). Signaling for a decision. ---
Time 99.77: Decision process sees failure. Choosing 'Overhaul'. Will take 25 hours to repair.
Time 124.77: Overhaul completed.
Time 124.77: Machine is 'New'. Operating for 176.08 hours.
Time 300.84: --- Machine FAILED (Minor_Failure). Signaling for a decision. ---
Time 300.84: Decision process sees failure. Choosing 'Overhaul'. Will take 25 hours to repair.
Time 325.84: Overhaul completed.
Time 325.84: Machine is 'New'. Operating for 9.11 hours.
Time 334.95: --- Machine FAILED (Minor_Failure). Signaling for a decision. ---
Time 334.95: Decision process sees failure. Choosing 'Overhaul'. Will take 25 hours to repair.
Time 359.95: Overhaul completed.
Time 359.95: Machine is 'New'. Operating for 8.42 hours.
Time 368.37: --- Machine FAILED (Minor_Failure). Signaling for a decision. ---
Time 368.37: Decision process sees failure. C

We use a very simple iterative cycle of `yield` statements together with two events that enable us to "hook into" the machine process, defer to a decision process and wait for a decision. This will be crucial for our interaction with `gym`.

There are multiple patterns of how you can achieve the same thing, you could also manually step through the events of the environment.
I think its idiomatic to simply yield a decision whenever one is needed, and otherwise let the simulation run.

## Wrapping it in a `gymnasium` environment

Now we need to interface this environment with `gymnasium`. The best course of action is to write a wrapper environment that implements the `gymnasium` interface.
Also, we have to remove the `decision_process` from our MachineEnv because we will replace this with the agent interaction.

We will also add more params to the `MachineSim` class, the durations and cost to overhaul.

In [60]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from collections import defaultdict

In [73]:
class MachineSim:
    def __init__(self, env: simpy.Environment):
        self.env = env

        self.economic_params = {
            'Quick_Fix': {'cost': 150, 'duration': 10},
            'Overhaul':  {'cost': 400, 'duration': 25}
        }
        self.machine_params = {
            'New': 100, 
            'Worn': 40 # Time to failure for each state
        }
        
        self.broken = self.env.event()
        self.repaired = self.env.event()
        self.env.process(self.machine_process())

    def machine_process(self):
        """Simulates the machine's lifecycle of operating and breaking."""
        machine_state = 'New'

        while True:
            # Operate until the next failure
            time_to_failure = random.expovariate(1.0 / self.machine_params[machine_state])
            print(f"Time {self.env.now:.2f}: Machine is '{machine_state}'. Operating for {time_to_failure:.2f} hours.")
            yield self.env.timeout(time_to_failure)
            
            # --- Failure Occurs ---
            failure_type = 'Minor_Failure' if machine_state == 'New' else 'Major_Failure'
            print(f"Time {self.env.now:.2f}: --- Machine FAILED ({failure_type}). Signaling for a decision. ---")
            self.broken.succeed(value=failure_type) # Signal that the machine is broken
            self.broken = self.env.event() # Reset the event for the next failure

            # Wait for the decision process to send back a repair action
            chosen_action = yield self.repaired
            print(f"Time {self.env.now:.2f}: {chosen_action} completed. Duration: {self.economic_params[chosen_action]['duration']} hours. Cost: {self.economic_params[chosen_action]['cost']}")
            
            # Update state based on the repair action
            machine_state = 'Worn' if chosen_action == 'Quick_Fix' else 'New'

In [74]:
import gymnasium as gym
from gymnasium import spaces

class SimpyManufacturingEnv(gym.Env):
    """
    This class wraps our SimPy MachineSim in a Gymnasium-compatible environment.
    """
    def __init__(self):
        super().__init__()
        
        self.action_map = {0: 'Quick_Fix', 1: 'Overhaul'}
        self.action_space = spaces.Discrete(len(self.action_map))
        
        # The agent sees two different states from which it must make a decision
        self.state_map = {'Minor_Failure': 0, 'Major_Failure': 1}
        self.observation_space = spaces.Discrete(len(self.state_map))

        self.sim_env = None
        self.machine_sim = None

    def _calculate_reward(self, action_str, duration):
        """Calculates reward as the negative of total cost for the cycle."""
        cost = self.machine_sim.economic_params[action_str]['cost'] + duration
        return -cost

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        # Start a new SimPy simulation
        self.sim_env = simpy.Environment()
        # Instantiate the machine simulation
        self.machine_sim = MachineSim(self.sim_env)
        
        # Run the simulation until the first breakdown
        self.sim_env.run(until=self.machine_sim.broken)
        
        # The first state the agent sees is "Minor Failure" as it came from "New"
        initial_state = 0 
        return initial_state, {} # Return initial state and empty info dict

    def step(self, action):
        action_str = self.action_map[action]
        start_time = self.sim_env.now
        
        # Simulate the repair time
        repair_duration = self.machine_sim.economic_params[action_str]['duration']
        self.sim_env.run(until=self.sim_env.timeout(repair_duration))
        
        # Signal repair is done
        self.machine_sim.repaired.succeed(value=action_str)
        self.machine_sim.repaired = self.sim_env.event()
        
        # Run until the next failure
        next_state_str = self.sim_env.run(until=self.machine_sim.broken)
        next_state = self.state_map[next_state_str]
        
        cycle_duration = self.sim_env.now - start_time
        reward = self._calculate_reward(action_str, cycle_duration)
        
        terminated = False
        truncated = False
        
        return next_state, reward, terminated, truncated, {}

We can do a simple naive Q-Learning update to predict the $Q(a|s)$ values:

In [91]:
# --- Q-Learning Training ---
print("Training Q-Learning agent on the SimPy MDP...")
env = SimpyManufacturingEnv()
num_steps = 5000 # More steps to learn the policy
learning_rate = 0.1
gamma = 0.99
epsilon = 0.1
Q = defaultdict(lambda: np.zeros(env.action_space.n))

state, info = env.reset()
for step in range(num_steps):
    action = np.argmax(Q[state]) if np.random.random() > epsilon else env.action_space.sample()
    print(f"Time {env.sim_env.now:.2f}: Decision process sees failure. Choosing '{env.action_map[action]}'.")
    next_state, reward, _, _, _ = env.step(action)
    td_target = reward + gamma * np.max(Q[next_state])
    Q[state][action] += learning_rate * (td_target - Q[state][action])
    state = next_state


print("Training finished.")

Training Q-Learning agent on the SimPy MDP...
Time 0.00: Machine is 'New'. Operating for 134.72 hours.
Time 134.72: --- Machine FAILED (Minor_Failure). Signaling for a decision. ---
Time 134.72: Decision process sees failure. Choosing 'Quick_Fix'.
Time 144.72: Quick_Fix completed. Duration: 10 hours. Cost: 150
Time 144.72: Machine is 'Worn'. Operating for 34.34 hours.
Time 179.07: --- Machine FAILED (Major_Failure). Signaling for a decision. ---
Time 179.07: Decision process sees failure. Choosing 'Quick_Fix'.
Time 189.07: Quick_Fix completed. Duration: 10 hours. Cost: 150
Time 189.07: Machine is 'Worn'. Operating for 28.34 hours.
Time 217.41: --- Machine FAILED (Major_Failure). Signaling for a decision. ---
Time 217.41: Decision process sees failure. Choosing 'Overhaul'.
Time 242.41: Overhaul completed. Duration: 25 hours. Cost: 400
Time 242.41: Machine is 'New'. Operating for 113.27 hours.
Time 355.67: --- Machine FAILED (Minor_Failure). Signaling for a decision. ---
Time 355.67: Dec

In [92]:
# --- Analyzing the State-Dependent Policy ---
print("\n--- Learned Q-Values ---")

# Policy for State 0 ('Minor_Failure')
q_minor = Q[0]
policy_minor = env.action_map[np.argmax(q_minor)]
print(f"State 'Minor_Failure': Q(Fix|Minor_Failure)={q_minor[0]:.2f}, Q(Overhaul|Minor_Failure)={q_minor[1]:.2f} -> Optimal Action: '{policy_minor}'")

# Policy for State 1 ('Major_Failure')
q_major = Q[1]
policy_major = env.action_map[np.argmax(q_major)]
print(f"State 'Major_Failure': Q(Fix|Major_Failure)={q_major[0]:.2f}, Q(Overhaul|Major_Failure)={q_major[1]:.2f} -> Optimal Action: '{policy_major}'")


--- Learned Q-Values ---
State 'Minor_Failure': Q(Fix|Minor_Failure)=-18902.42, Q(Overhaul|Minor_Failure)=-18890.17 -> Optimal Action: 'Overhaul'
State 'Major_Failure': Q(Fix|Major_Failure)=-19058.40, Q(Overhaul|Major_Failure)=-19097.30 -> Optimal Action: 'Quick_Fix'


We see that the agent correctly learned that the more expensive `overhaul` action is the right to pick if the failure is `minor` because a `quick fix` would just restore it to `worn` again. The Q-values are high, because we consider the full future with a relatively high (low discount) $\gamma$.

## Using `stablebaselines3` on our environment
Because the environment is now a `gymnasium` environment, we can use `stablebaselines3` to train an agent on it! We can use the `check_env` functionality to test compatability first which just samples a couple of actions.

In [ ]:
from stable_baselines3.common.env_checker import check_env 
check_env(SimpyManufacturingEnv())

Time 0.00: Machine is 'New'. Operating for 282.41 hours.
Time 282.41: --- Machine FAILED (Minor_Failure). Signaling for a decision. ---
Time 0.00: Machine is 'New'. Operating for 100.06 hours.
Time 100.06: --- Machine FAILED (Minor_Failure). Signaling for a decision. ---
Time 125.06: Overhaul completed. Duration: 25 hours. Cost: 400
Time 125.06: Machine is 'New'. Operating for 58.28 hours.
Time 183.34: --- Machine FAILED (Minor_Failure). Signaling for a decision. ---
Time 0.00: Machine is 'New'. Operating for 8.93 hours.
Time 8.93: --- Machine FAILED (Minor_Failure). Signaling for a decision. ---
Time 18.93: Quick_Fix completed. Duration: 10 hours. Cost: 150
Time 18.93: Machine is 'Worn'. Operating for 9.10 hours.
Time 28.03: --- Machine FAILED (Major_Failure). Signaling for a decision. ---
Time 53.03: Overhaul completed. Duration: 25 hours. Cost: 400
Time 53.03: Machine is 'New'. Operating for 148.64 hours.
Time 201.67: --- Machine FAILED (Minor_Failure). Signaling for a decision. ---

In [82]:
from stable_baselines3 import DQN

env = SimpyManufacturingEnv()
model = DQN("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=5000)
env.close()

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Time 0.00: Machine is 'New'. Operating for 70.33 hours.
Time 70.33: --- Machine FAILED (Minor_Failure). Signaling for a decision. ---
Time 80.33: Quick_Fix completed. Duration: 10 hours. Cost: 150
Time 80.33: Machine is 'Worn'. Operating for 41.49 hours.
Time 121.82: --- Machine FAILED (Major_Failure). Signaling for a decision. ---
Time 146.82: Overhaul completed. Duration: 25 hours. Cost: 400
Time 146.82: Machine is 'New'. Operating for 84.96 hours.
Time 231.77: --- Machine FAILED (Minor_Failure). Signaling for a decision. ---
Time 256.77: Overhaul completed. Duration: 25 hours. Cost: 400
Time 256.77: Machine is 'New'. Operating for 101.76 hours.
Time 358.53: --- Machine FAILED (Minor_Failure). Signaling for a decision. ---
Time 368.53: Quick_Fix completed. Duration: 10 hours. Cost: 150
Time 368.53: Machine is 'Worn'. Operating for 19.78 hours.
Time 388.31: --- Machine FAILED (Major_Failure).

Voila, we have created a `gym` wrapper for our `simpy` environment, that enables us to use agents from `sb3`!